In [1]:
import polars as pl
import json

pathologies_path = r'C:\Users\Jacey Ong\Documents\Personal Stuff\Coding Projects\Automated Differential Diagnosis via Information Theory\ddx_plus_english\release_conditions.json'

with open(pathologies_path) as f:
    data = json.load(f)

pathologies_json =[]

for _, val in data.items():
    pathologies_json.append({
        'pathology': val.get('condition_name'),
        'icd10_id': val.get('icd10-id'),
        'symptoms': val.get('symptoms'),
        'antecedents': val.get('antecedents'),
        'severity': val.get('severity')
    })

for pathology in pathologies_json:
    symptoms = []
    antecedents = []
    for s, val in pathology['symptoms'].items():
        symptoms.append(s)
    for a, val in pathology['antecedents'].items():
        antecedents.append(a)

    pathology['symptoms'] = symptoms
    pathology['antecedents'] = antecedents

pathologies_json

pathologies_df = pl.DataFrame(pathologies_json)


In [2]:
pathology = pathologies_df.select(['pathology','icd10_id','severity'])
pathology

pathology,icd10_id,severity
str,str,i64
"""Spontaneous pneumothorax""","""J93""",2
"""Cluster headache""","""g44.009""",3
"""Boerhaave""","""K22.3""",2
"""Spontaneous rib fracture""","""S22.9""",3
"""GERD""","""K21""",3
…,…,…
"""Possible NSTEMI / STEMI""","""I21""",1
"""Sarcoidosis""","""d86""",4
"""Pancreatic neoplasm""","""c25""",3


In [20]:
from sqlalchemy import create_engine 
# need to pip install mysql-connector-python first in terminal

user = "root"
password = "root"
host = "127.0.0.1"
port = 3306
database = "ddxplus_db"

url = f"mysql://{user}:{password}@{host}:{port}/{database}"

engine = create_engine(url)

In [51]:
pathology.write_database(
    table_name='pathology_tbl',
    connection = engine,
    if_table_exists='append',
    engine = 'sqlalchemy'
)

49

In [3]:
pathology_evidences = pathologies_df.select(['pathology','symptoms','antecedents'])
pathology_evidences = pathology_evidences.with_columns(pl.concat_list('symptoms','antecedents').alias('evidence')).drop('symptoms','antecedents')
pathology_evidences = pathology_evidences.explode('evidence')
pathology_evidences

pathology,evidence
str,str
"""Spontaneous pneumothorax""","""E_55"""
"""Spontaneous pneumothorax""","""E_53"""
"""Spontaneous pneumothorax""","""E_57"""
"""Spontaneous pneumothorax""","""E_54"""
"""Spontaneous pneumothorax""","""E_59"""
…,…
"""Pericarditis""","""E_66"""
"""Pericarditis""","""E_155"""
"""Pericarditis""","""E_0"""


In [64]:
pathology_evidences.write_database(
    table_name='pathology_evidences_tbl',
    connection = engine,
    if_table_exists='append',
    engine = 'sqlalchemy'
)

888

In [4]:
evidences_path = r'C:\Users\Jacey Ong\Documents\Personal Stuff\Coding Projects\Automated Differential Diagnosis via Information Theory\ddx_plus_english\release_evidences.json'

with open(evidences_path) as f:
    evidences_raw = json.load(f)

evidences_json = []

for _, val in evidences_raw.items():
    evidences_json.append(
        {
            'evidence_code': val.get('name'),
            'root_evidence': val.get('code_question'),
            'question': val.get('question_en'),
            'is_antecedent': val.get('is_antecedent'),
            'evidence_type': val.get('data_type'),
            'default_value': val.get('default_value'),
            'possible_values': val.get('possible-values'),
            'value_meaning': val.get('value_meaning')
        }
    )

evidences_df = pl.DataFrame(evidences_json)
evidences_df

shape: (223, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ evidence_c ┆ root_evide ┆ question   ┆ is_antece ┆ evidence_ ┆ default_v ┆ possible_ ┆ value_mea │
│ ode        ┆ nce        ┆ ---        ┆ dent      ┆ type      ┆ alue      ┆ values    ┆ ning      │
│ ---        ┆ ---        ┆ str        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆ str        ┆            ┆ bool      ┆ str       ┆ str       ┆ list[str] ┆ struct[18 │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ 8]        │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ E_91       ┆ E_91       ┆ Do you     ┆ false     ┆ B         ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ have a     ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ fever      ┆           ┆           ┆           ┆           ┆ ll,null,n │
│            ┆            ┆ (either    ┆           ┆           ┆           ┆           ┆ ull…      │
│            ┆            ┆ fe…        ┆           ┆           ┆           ┆           ┆           │
│ E_55       ┆ E_53       ┆ Do you     ┆ false     ┆ M         ┆ V_123     ┆ ["V_123", ┆ {{"nulle  │
│            ┆            ┆ feel pain  ┆           ┆           ┆           ┆ "V_14", … ┆ part","no │
│            ┆            ┆ somewhere? ┆           ┆           ┆           ┆ "V_197"]  ┆ where"},{ │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ "ai…      │
│ E_53       ┆ E_53       ┆ Do you     ┆ false     ┆ B         ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ have pain  ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ somewhere, ┆           ┆           ┆           ┆           ┆ ll,null,n │
│            ┆            ┆ re…        ┆           ┆           ┆           ┆           ┆ ull…      │
│ E_57       ┆ E_53       ┆ Does the   ┆ false     ┆ M         ┆ V_123     ┆ ["V_123", ┆ {{"nulle  │
│            ┆            ┆ pain       ┆           ┆           ┆           ┆ "V_14", … ┆ part","no │
│            ┆            ┆ radiate to ┆           ┆           ┆           ┆ "V_197"]  ┆ where"},{ │
│            ┆            ┆ anoth…     ┆           ┆           ┆           ┆           ┆ "ai…      │
│ E_54       ┆ E_53       ┆ Characteri ┆ false     ┆ M         ┆ V_11      ┆ ["V_11",  ┆ {null,nul │
│            ┆            ┆ ze your    ┆           ┆           ┆           ┆ "V_71", … ┆ l,null,nu │
│            ┆            ┆ pain:      ┆           ┆           ┆           ┆ "V_198"]  ┆ ll,null,n │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ ull…      │
│ …          ┆ …          ┆ …          ┆ …         ┆ …         ┆ …         ┆ …         ┆ …         │
│ E_195      ┆ E_195      ┆ Do you     ┆ true      ┆ B         ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ live in    ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ the        ┆           ┆           ┆           ┆           ┆ ll,null,n │
│            ┆            ┆ suburbs?   ┆           ┆           ┆           ┆           ┆ ull…      │
│ E_183      ┆ E_183      ┆ Do you     ┆ true      ┆ B         ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ live in a  ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ rural      ┆           ┆           ┆           ┆           ┆ ll,null,n │
│            ┆            ┆ area?      ┆           ┆           ┆           ┆           ┆ ull…      │
│ E_224      ┆ E_224      ┆ Do you     ┆ true      ┆ B         ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ have       ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ family     ┆     

In [5]:
evidences_df = evidences_df.with_columns(evidence_type = pl.col('evidence_type').replace({'B':'binary','C':'categorical','M':'multichoice'}))
evidences_df

shape: (223, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ evidence_c ┆ root_evide ┆ question   ┆ is_antece ┆ evidence_ ┆ default_v ┆ possible_ ┆ value_mea │
│ ode        ┆ nce        ┆ ---        ┆ dent      ┆ type      ┆ alue      ┆ values    ┆ ning      │
│ ---        ┆ ---        ┆ str        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆ str        ┆            ┆ bool      ┆ str       ┆ str       ┆ list[str] ┆ struct[18 │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ 8]        │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ E_91       ┆ E_91       ┆ Do you     ┆ false     ┆ binary    ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ have a     ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ fever      ┆           ┆           ┆           ┆           ┆ ll,null,n │
│            ┆            ┆ (either    ┆           ┆           ┆           ┆           ┆ ull…      │
│            ┆            ┆ fe…        ┆           ┆           ┆           ┆           ┆           │
│ E_55       ┆ E_53       ┆ Do you     ┆ false     ┆ multichoi ┆ V_123     ┆ ["V_123", ┆ {{"nulle  │
│            ┆            ┆ feel pain  ┆           ┆ ce        ┆           ┆ "V_14", … ┆ part","no │
│            ┆            ┆ somewhere? ┆           ┆           ┆           ┆ "V_197"]  ┆ where"},{ │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ "ai…      │
│ E_53       ┆ E_53       ┆ Do you     ┆ false     ┆ binary    ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ have pain  ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ somewhere, ┆           ┆           ┆           ┆           ┆ ll,null,n │
│            ┆            ┆ re…        ┆           ┆           ┆           ┆           ┆ ull…      │
│ E_57       ┆ E_53       ┆ Does the   ┆ false     ┆ multichoi ┆ V_123     ┆ ["V_123", ┆ {{"nulle  │
│            ┆            ┆ pain       ┆           ┆ ce        ┆           ┆ "V_14", … ┆ part","no │
│            ┆            ┆ radiate to ┆           ┆           ┆           ┆ "V_197"]  ┆ where"},{ │
│            ┆            ┆ anoth…     ┆           ┆           ┆           ┆           ┆ "ai…      │
│ E_54       ┆ E_53       ┆ Characteri ┆ false     ┆ multichoi ┆ V_11      ┆ ["V_11",  ┆ {null,nul │
│            ┆            ┆ ze your    ┆           ┆ ce        ┆           ┆ "V_71", … ┆ l,null,nu │
│            ┆            ┆ pain:      ┆           ┆           ┆           ┆ "V_198"]  ┆ ll,null,n │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ ull…      │
│ …          ┆ …          ┆ …          ┆ …         ┆ …         ┆ …         ┆ …         ┆ …         │
│ E_195      ┆ E_195      ┆ Do you     ┆ true      ┆ binary    ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ live in    ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ the        ┆           ┆           ┆           ┆           ┆ ll,null,n │
│            ┆            ┆ suburbs?   ┆           ┆           ┆           ┆           ┆ ull…      │
│ E_183      ┆ E_183      ┆ Do you     ┆ true      ┆ binary    ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ live in a  ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ rural      ┆           ┆           ┆           ┆           ┆ ll,null,n │
│            ┆            ┆ area?      ┆           ┆           ┆           ┆           ┆ ull…      │
│ E_224      ┆ E_224      ┆ Do you     ┆ true      ┆ binary    ┆ 0         ┆ []        ┆ {null,nul │
│            ┆            ┆ have       ┆           ┆           ┆           ┆           ┆ l,null,nu │
│            ┆            ┆ family     ┆     

In [6]:
numerical_ev = evidences_df.select(pl.exclude(['possible_values','value_meaning'])).filter(pl.col('evidence_type')=='categorical',~pl.col('default_value').str.contains('V_'))
numerical_ev = numerical_ev.cast({'default_value':pl.Int8})
numerical_ev = numerical_ev.rename({'default_value':'default_value_num'})
numerical_ev

evidence_code,root_evidence,question,is_antecedent,evidence_type,default_value_num
str,str,str,bool,str,i8
"""E_59""","""E_53""","""How fast did the pain appear?""",false,"""categorical""",0
"""E_56""","""E_53""","""How intense is the pain?""",false,"""categorical""",0
"""E_58""","""E_53""","""How precisely is the pain loca…",false,"""categorical""",0
"""E_134""","""E_129""","""How intense is the pain caused…",false,"""categorical""",0
"""E_132""","""E_129""","""Is the rash swollen?""",false,"""categorical""",0
"""E_136""","""E_129""","""How severe is the itching?""",false,"""categorical""",0


In [7]:
binary_ev = evidences_df.select(pl.exclude(['possible_values','value_meaning'])).filter(evidence_type = 'binary')
binary_ev = binary_ev.cast({'default_value':pl.Int8})
binary_ev = binary_ev.rename({'default_value':'default_value_bool'})
binary_ev

evidence_code,root_evidence,question,is_antecedent,evidence_type,default_value_bool
str,str,str,bool,str,i8
"""E_91""","""E_91""","""Do you have a fever (either fe…",false,"""binary""",0
"""E_53""","""E_53""","""Do you have pain somewhere, re…",false,"""binary""",0
"""E_159""","""E_159""","""Did you lose consciousness?""",false,"""binary""",0
"""E_129""","""E_129""","""Do you have any lesions, redne…",false,"""binary""",0
"""E_154""","""E_154""","""Is your skin much paler than u…",false,"""binary""",0
…,…,…,…,…,…
"""E_195""","""E_195""","""Do you live in the suburbs?""",true,"""binary""",0
"""E_183""","""E_183""","""Do you live in a rural area?""",true,"""binary""",0
"""E_224""","""E_224""","""Do you have family members who…",true,"""binary""",0


In [8]:
enby_ev = evidences_df.select(pl.exclude(['possible_values','value_meaning'])).filter(pl.col('default_value').str.contains('V_'))
enby_ev = enby_ev.rename({'default_value':'default_value_str'})
enby_ev

evidence_code,root_evidence,question,is_antecedent,evidence_type,default_value_str
str,str,str,bool,str,str
"""E_55""","""E_53""","""Do you feel pain somewhere?""",false,"""multichoice""","""V_123"""
"""E_57""","""E_53""","""Does the pain radiate to anoth…",false,"""multichoice""","""V_123"""
"""E_54""","""E_53""","""Characterize your pain:""",false,"""multichoice""","""V_11"""
"""E_133""","""E_129""","""Where is the affected region l…",false,"""multichoice""","""V_123"""
"""E_130""","""E_129""","""What color is the rash?""",false,"""categorical""","""V_11"""
"""E_135""","""E_129""","""Is the lesion (or are the lesi…",false,"""categorical""","""V_10"""
"""E_131""","""E_129""","""Do your lesions peel off?""",false,"""categorical""","""V_10"""
"""E_152""","""E_151""","""Where is the swelling located?""",false,"""multichoice""","""V_123"""
"""E_204""","""E_204""","""Have you traveled out of the c…",true,"""categorical""","""V_10"""


In [9]:
evidences = pl.concat([binary_ev, numerical_ev, enby_ev],how='diagonal')
evidences

evidence_code,root_evidence,question,is_antecedent,evidence_type,default_value_bool,default_value_num,default_value_str
str,str,str,bool,str,i8,i8,str
"""E_91""","""E_91""","""Do you have a fever (either fe…",false,"""binary""",0,null,null
"""E_53""","""E_53""","""Do you have pain somewhere, re…",false,"""binary""",0,null,null
"""E_159""","""E_159""","""Did you lose consciousness?""",false,"""binary""",0,null,null
"""E_129""","""E_129""","""Do you have any lesions, redne…",false,"""binary""",0,null,null
"""E_154""","""E_154""","""Is your skin much paler than u…",false,"""binary""",0,null,null
…,…,…,…,…,…,…,…
"""E_130""","""E_129""","""What color is the rash?""",false,"""categorical""",null,null,"""V_11"""
"""E_135""","""E_129""","""Is the lesion (or are the lesi…",false,"""categorical""",null,null,"""V_10"""
"""E_131""","""E_129""","""Do your lesions peel off?""",false,"""categorical""",null,null,"""V_10"""


In [98]:
evidences.write_database(
    'evidences_tbl',
    if_table_exists='append',
    connection=engine,
    engine = 'sqlalchemy'
)

223

In [52]:
ev_val_df = evidences_df.select(['evidence_code','evidence_type','possible_values'])
ev_val_df = ev_val_df.with_columns(possible_values = pl.when(evidence_type = 'binary').then(pl.lit([0,1]))\
                                   .otherwise(pl.col('possible_values')))
ev_val_df = ev_val_df.explode('possible_values')
ev_val_df = ev_val_df.with_columns(pl.concat_str([pl.col('evidence_code'), pl.col('possible_values')],separator="_@_").alias('evidence_value'))

ev_val_bool = ev_val_df.filter(evidence_type = 'binary')\
                        .cast({'possible_values':pl.Int8}).rename({'possible_values':'possible_values_bool'})
ev_val_num = ev_val_df.filter(pl.col('evidence_type')=='categorical',~pl.col('possible_values').str.contains('V_'))\
            .cast({'possible_values':pl.Int8}).rename({'possible_values':'possible_values_num'})
ev_val_str = ev_val_df.filter(pl.col('possible_values').str.contains('V_')).rename({'possible_values':'possible_values_str'})


In [54]:
ev_val = pl.concat([ev_val_bool, ev_val_num, ev_val_str],how='diagonal') 
ev_val


evidence_code,evidence_type,possible_values_bool,evidence_value,possible_values_num,possible_values_str
str,str,i8,str,i8,str
"""E_91""","""binary""",0,"""E_91_@_0""",null,null
"""E_91""","""binary""",1,"""E_91_@_1""",null,null
"""E_53""","""binary""",0,"""E_53_@_0""",null,null
"""E_53""","""binary""",1,"""E_53_@_1""",null,null
"""E_159""","""binary""",0,"""E_159_@_0""",null,null
…,…,…,…,…,…
"""E_204""","""categorical""",null,"""E_204_@_V_6""",null,"""V_6"""
"""E_204""","""categorical""",null,"""E_204_@_V_7""",null,"""V_7"""
"""E_204""","""categorical""",null,"""E_204_@_V_8""",null,"""V_8"""


In [56]:
ev_val.write_database(
    'evidence_values_tbl',
    if_table_exists='append',
    connection=engine,
    engine='sqlalchemy'
)

1180

In [57]:
meaning_dict ={
    'value_str':[],
    'value_meaning':[],
    'associated_evidence':[],
}

for ev in evidences_json:
    if ev['value_meaning'] == {}:
        continue

    for k, v in ev['value_meaning'].items():
        meaning_dict['value_str'].append(k)
        meaning_dict['value_meaning'].append(v['en'])
        meaning_dict['associated_evidence'].append(ev['evidence_code'])

meaning_df = pl.DataFrame(meaning_dict)
meaning_df = meaning_df.unique(maintain_order=True)
meaning_df

value_str,value_meaning,associated_evidence
str,str,str
"""V_123""","""nowhere""","""E_55"""
"""V_14""","""iliac wing(R)""","""E_55"""
"""V_15""","""iliac wing(L)""","""E_55"""
"""V_16""","""groin(R)""","""E_55"""
"""V_17""","""groin(L)""","""E_55"""
…,…,…
"""V_6""","""Asia""","""E_204"""
"""V_7""","""South East Asia""","""E_204"""
"""V_8""","""Caraibes""","""E_204"""


In [ ]:
value_freq = meaning_df.group_by('value_str').len()
value_freq.filter(pl.col('len') == 1)

value_str,len
str,u32
"""V_1""",1
"""V_184""",1
"""V_191""",1
"""V_156""",1
"""V_181""",1
…,…
"""V_193""",1
"""V_86""",1
"""V_154""",1


In [58]:
meaning_df.write_database(
    'string_values_tbl',
    if_table_exists='append',
    connection=engine,
    engine='sqlalchemy'
)

698

In [16]:
evidences_json[1]['evidence_code']

'E_55'